In [13]:
import osmium
import pickle
import math
from collections import defaultdict

KEEP_HIGHWAY = {
    "motorway", "motorway_link",
    "trunk", "trunk_link",
    "primary", "primary_link",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link",
    "residential",
    "unclassified",
}

TEST_POINTS = [
    ("DEPOT", 51.325918, 12.011645),

    ("C1", 51.282499, 9.526584),
    ("C2", 51.623978, 9.936278),
    ("C3", 51.488659, 9.007833),
    ("C4", 51.287606, 8.873039),
    ("C5", 51.029800, 9.421600),
    ("C6", 50.861352, 9.721422),
    ("C7", 50.980433, 10.287236),
    ("C8", 51.200747, 10.018705),
    ("C9", 51.390280, 10.125665),
    ("C10", 51.432132, 9.647465),
]

PAD = 0.2

LATS = [p[1] for p in TEST_POINTS]
LONS = [p[2] for p in TEST_POINTS]

LAT_MIN = min(LATS) - PAD
LAT_MAX = max(LATS) + PAD
LON_MIN = min(LONS) - PAD
LON_MAX = max(LONS) + PAD


def in_bbox(lat, lon):
    return LAT_MIN <= lat <= LAT_MAX and LON_MIN <= lon <= LON_MAX


def haversine(lat1, lon1, lat2, lon2):
    R = 6371000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c


class NodeCounter(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.node_count = defaultdict(int)
        self.way_count = 0
        self.kept_way_count = 0

    def way(self, w):
        self.way_count += 1

        if self.way_count % 200000 == 0:
            print(f"  Pass 1 ... {self.way_count:,} Ways geprüft, {self.kept_way_count:,} behalten")

        if w.tags.get("highway") not in KEEP_HIGHWAY:
            return

        try:
            nodes = list(w.nodes)
            if len(nodes) < 2:
                return

            keep = False
            for n in nodes:
                if n.location.valid() and in_bbox(n.location.lat, n.location.lon):
                    keep = True
                    break

            if not keep:
                return

            self.kept_way_count += 1

            for node in nodes:
                self.node_count[node.ref] += 1

        except Exception:
            return


class CompressedGraphBuilder(osmium.SimpleHandler):
    def __init__(self, junction_nodes):
        super().__init__()
        self.junction_nodes = junction_nodes
        self.node_coords = {}
        self.edges = []
        self.arc_id = 0
        self.way_count = 0
        self.kept_way_count = 0

    def way(self, w):
        self.way_count += 1

        if self.way_count % 200000 == 0:
            print(f"  Pass 2 ... {self.way_count:,} Ways geprüft, {self.kept_way_count:,} behalten, {len(self.edges):,} Kanten")

        hwy = w.tags.get("highway")
        if hwy not in KEEP_HIGHWAY:
            return

        nodes = list(w.nodes)
        if len(nodes) < 2:
            return

        oneway = w.tags.get("oneway", "no")
        name = w.tags.get("name", "")
        tunnel = w.tags.get("tunnel", "no")

        coords = {}
        keep = False

        for n in nodes:
            try:
                lat = n.location.lat
                lon = n.location.lon
                coords[n.ref] = (lat, lon)
                if in_bbox(lat, lon):
                    keep = True
            except Exception:
                return

        if not keep:
            return

        self.kept_way_count += 1

        segment_start = nodes[0].ref
        segment_dist = 0.0
        segment_coords = [coords[segment_start]]

        for i in range(len(nodes) - 1):
            n1, n2 = nodes[i].ref, nodes[i + 1].ref

            lat1, lon1 = coords[n1]
            lat2, lon2 = coords[n2]
            segment_dist += haversine(lat1, lon1, lat2, lon2)
            segment_coords.append(coords[n2])

            is_junction = (n2 in self.junction_nodes or i == len(nodes) - 2)

            if is_junction:
                u, v = segment_start, n2

                self.node_coords[u] = coords[u]
                self.node_coords[v] = coords[v]

                edge_data = {
                    "arc_id": self.arc_id,
                    "u": u,
                    "v": v,
                    "dist": segment_dist,   # Meter
                    "highway": hwy,
                    "name": name,
                    "tunnel": tunnel,
                    "geometry": segment_coords.copy(),
                }
                self.edges.append(edge_data)
                self.arc_id += 1

                if oneway in ("", "no", "false", "0"):
                    edge_data_rev = {
                        "arc_id": self.arc_id,
                        "u": v,
                        "v": u,
                        "dist": segment_dist,
                        "highway": hwy,
                        "name": name,
                        "tunnel": tunnel,
                        "geometry": segment_coords[::-1],
                    }
                    self.edges.append(edge_data_rev)
                    self.arc_id += 1

                segment_start = n2
                segment_dist = 0.0
                segment_coords = [coords[n2]]


def build_compressed_graph(pbf_path, output_path):
    print("Bounding Box Test:")
    print(f"  lat: {LAT_MIN:.4f} bis {LAT_MAX:.4f}")
    print(f"  lon: {LON_MIN:.4f} bis {LON_MAX:.4f}")
    print(f"  PAD: {PAD}")

    print("\nPass 1: Knoten zählen...")
    counter = NodeCounter()
    counter.apply_file(pbf_path, locations=True)

    junction_nodes = {node for node, count in counter.node_count.items() if count != 2}
    print(f"  → {len(junction_nodes):,} Junction-Knoten")

    print("\nPass 2: Graph aufbauen...")
    builder = CompressedGraphBuilder(junction_nodes)
    builder.apply_file(pbf_path, locations=True)

    data = {
        "nodes": list(builder.node_coords.keys()),
        "node_coords": builder.node_coords,
        "arcs": builder.edges,
        "bbox": {
            "lat_min": LAT_MIN,
            "lat_max": LAT_MAX,
            "lon_min": LON_MIN,
            "lon_max": LON_MAX,
            "pad": PAD,
        }
    }

    with open(output_path, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

    print("\n✅ Graph fertig!")
    print(f"  Knoten: {len(data['nodes']):,}")
    print(f"  Kanten: {len(data['arcs']):,}")

    return data


if __name__ == "__main__":
    build_compressed_graph(
        r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\germany-latest.osm.pbf",
        "graph_small.pkl"
    )


Bounding Box Test:
  lat: 50.6614 bis 51.8240
  lon: 8.6730 bis 12.2116
  PAD: 0.2

Pass 1: Knoten zählen...
  Pass 1 ... 200,000 Ways geprüft, 3,194 behalten
  Pass 1 ... 400,000 Ways geprüft, 8,788 behalten
  Pass 1 ... 600,000 Ways geprüft, 13,097 behalten
  Pass 1 ... 800,000 Ways geprüft, 18,776 behalten
  Pass 1 ... 1,000,000 Ways geprüft, 25,410 behalten
  Pass 1 ... 1,200,000 Ways geprüft, 31,394 behalten
  Pass 1 ... 1,400,000 Ways geprüft, 37,050 behalten
  Pass 1 ... 1,600,000 Ways geprüft, 42,845 behalten
  Pass 1 ... 1,800,000 Ways geprüft, 47,428 behalten
  Pass 1 ... 2,000,000 Ways geprüft, 51,068 behalten
  Pass 1 ... 2,200,000 Ways geprüft, 54,476 behalten
  Pass 1 ... 2,400,000 Ways geprüft, 58,675 behalten
  Pass 1 ... 2,600,000 Ways geprüft, 62,046 behalten
  Pass 1 ... 2,800,000 Ways geprüft, 66,016 behalten
  Pass 1 ... 3,000,000 Ways geprüft, 69,836 behalten
  Pass 1 ... 3,200,000 Ways geprüft, 73,802 behalten
  Pass 1 ... 3,400,000 Ways geprüft, 77,427 behalten


Unfälle

In [14]:
import pickle
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
from shapely.strtree import STRtree

def load_accidents(csv_path):
    df = pd.read_csv(csv_path, sep=";", low_memory=False)
    df.columns = df.columns.str.strip()

    for col in ["XGCSWGS84", "YGCSWGS84"]:
        df[col] = df[col].astype(str).str.replace(",", ".").astype(float)

    df = df.dropna(subset=["XGCSWGS84", "YGCSWGS84"]).copy()

    df["weight"] = df["UKATEGORIE"].map({1: 5, 2: 3, 3: 1}).fillna(1)

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["XGCSWGS84"], df["YGCSWGS84"]),
        crs="EPSG:4326"
    )
    return gdf


def build_road_geometries(graph_data):
    rows = []

    for arc in graph_data["arcs"]:
        coords = arc.get("geometry", [])

        if len(coords) >= 2:
            line_coords = [(lon, lat) for lat, lon in coords]
            geom = LineString(line_coords)
        else:
            u, v = arc["u"], arc["v"]
            c1 = graph_data["node_coords"].get(u)
            c2 = graph_data["node_coords"].get(v)
            if c1 and c2:
                geom = LineString([(c1[1], c1[0]), (c2[1], c2[0])])
            else:
                continue

        rows.append({
            "arc_id": arc["arc_id"],
            "u": arc["u"],
            "v": arc["v"],
            "dist": arc["dist"],
            "highway": arc.get("highway", ""),
            "name": arc.get("name", ""),
            "tunnel": arc.get("tunnel", "no"),
            "geometry": geom,
        })

    return gpd.GeoDataFrame(rows, crs="EPSG:4326")


def map_accidents_to_edges(gdf_roads, gdf_accidents, buffer_m=30):
    crs_metric = "EPSG:32632"

    roads = gdf_roads.to_crs(crs_metric).copy()
    accidents = gdf_accidents.to_crs(crs_metric).copy()

    roads["length_m"] = roads.geometry.length

    road_geoms = roads.geometry.values
    tree = STRtree(road_geoms)

    accident_assignments = []

    for idx, acc in accidents.iterrows():
        point = acc.geometry
        candidates = tree.query(point.buffer(buffer_m))

        if len(candidates) == 0:
            continue

        min_dist = float("inf")
        best_idx = None

        for cand_idx in candidates:
            dist = point.distance(road_geoms[cand_idx])
            if dist < min_dist:
                min_dist = dist
                best_idx = cand_idx

        if best_idx is not None and min_dist <= buffer_m:
            accident_assignments.append({
                "accident_idx": idx,
                "arc_id": roads.iloc[best_idx]["arc_id"],
                "distance": min_dist,
                "weight": acc["weight"],
            })

    assignments_df = pd.DataFrame(accident_assignments)

    if len(assignments_df) > 0:
        agg = assignments_df.groupby("arc_id").agg(
            accidents=("arc_id", "count"),
            weighted_accidents=("weight", "sum"),
            avg_distance=("distance", "mean"),
        ).reset_index()
    else:
        agg = pd.DataFrame(columns=["arc_id", "accidents", "weighted_accidents", "avg_distance"])

    roads = roads.merge(agg, on="arc_id", how="left")
    roads["accidents"] = roads["accidents"].fillna(0)
    roads["weighted_accidents"] = roads["weighted_accidents"].fillna(0)

    roads["accident_score"] = roads["weighted_accidents"] / (roads["length_m"] + 1)

    max_score = roads["accident_score"].max()
    if max_score > 0:
        roads["accident_score_norm"] = roads["accident_score"] / max_score
    else:
        roads["accident_score_norm"] = 0

    print("✅ Unfälle gemappt:")
    print(f"   Gesamt: {len(gdf_accidents):,}")
    print(f"   Zugeordnet: {len(assignments_df):,}")
    print(f"   Kanten mit Unfällen: {(roads['accidents'] > 0).sum():,}")

    return roads


if __name__ == "__main__":
    GRAPH_PATH = r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Routenplanung_Gefahrguttransport\models\Berlin\graph_small.pkl"
    ACCIDENTS_CSV = r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\Umwelftfreundliche_Routenplanung\data\raw\Accidents.csv"
    OUTPUT_PATH = "germany_graph_with_accidents_small.pkl"

    with open(GRAPH_PATH, "rb") as f:
        graph = pickle.load(f)
    print(f"Graph: {len(graph['arcs']):,} Kanten")

    gdf_roads = build_road_geometries(graph)
    print(f"Geometrien: {len(gdf_roads):,}")

    gdf_accidents = load_accidents(ACCIDENTS_CSV)
    print(f"Unfälle: {len(gdf_accidents):,}")

    roads_with_accidents = map_accidents_to_edges(gdf_roads, gdf_accidents, buffer_m=30)

    export_data = {
        "nodes": graph["nodes"],
        "node_coords": graph["node_coords"],
        "edges": roads_with_accidents[[
            "arc_id", "u", "v", "dist", "highway", "name", "tunnel",
            "length_m", "accidents", "weighted_accidents",
            "accident_score", "accident_score_norm"
        ]].to_dict("records"),
    }

    with open(OUTPUT_PATH, "wb") as f:
        pickle.dump(export_data, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"\n✅ Export: {OUTPUT_PATH}")


Graph: 3,348,685 Kanten
Geometrien: 3,348,685
Unfälle: 268,519
✅ Unfälle gemappt:
   Gesamt: 268,519
   Zugeordnet: 13,189
   Kanten mit Unfällen: 12,523

✅ Export: germany_graph_with_accidents_small.pkl


Bevölkerung

In [15]:
import pickle
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import LineString

def map_population_to_edges(gdf_roads, pop_gpkg_path, buffer_m=50):
    gdf_pop = gpd.read_file(pop_gpkg_path)
    gdf_pop = gdf_pop[["T", "geometry"]].copy()
    gdf_pop = gdf_pop[gdf_pop["T"] > 0]
    gdf_pop = gdf_pop.rename(columns={"T": "population"})

    print(f"Bevölkerungszellen: {len(gdf_pop):,}")

    crs_metric = "EPSG:3035"

    roads = gdf_roads.to_crs(crs_metric).copy()
    pop = gdf_pop.to_crs(crs_metric).copy()

    pop["cell_area"] = pop.geometry.area

    roads["geometry_original"] = roads.geometry
    roads["geometry"] = roads.geometry.buffer(buffer_m)

    overlay = gpd.overlay(roads, pop, how="intersection")

    if len(overlay) > 0:
        overlay["overlap_area"] = overlay.geometry.area
        overlay["pop_fraction"] = overlay["overlap_area"] / overlay["cell_area"]
        overlay["assigned_pop"] = overlay["population"] * overlay["pop_fraction"]

        pop_per_edge = overlay.groupby("arc_id").agg(
            total_pop=("assigned_pop", "sum"),
            cells_touched=("arc_id", "count"),
        ).reset_index()
    else:
        pop_per_edge = pd.DataFrame(columns=["arc_id", "total_pop", "cells_touched"])

    roads["geometry"] = roads["geometry_original"]
    roads = roads.drop(columns=["geometry_original"])

    roads = roads.merge(pop_per_edge, on="arc_id", how="left")
    roads["total_pop"] = roads["total_pop"].fillna(0)

    roads["length_m"] = roads.geometry.length
    roads["pop_per_m"] = roads["total_pop"] / (roads["length_m"] + 1)

    max_pop = roads["pop_per_m"].max()
    if max_pop > 0:
        roads["pop_score_norm"] = roads["pop_per_m"] / max_pop
    else:
        roads["pop_score_norm"] = 0

    print("✅ Bevölkerung gemappt:")
    print(f"   Kanten mit Bevölkerung > 0: {(roads['total_pop'] > 0).sum():,}")
    print(f"   Max Pop/m: {roads['pop_per_m'].max():.2f}")

    return roads


def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c


def map_charging_to_nodes(node_coords, charging_csv_path, min_power_kw=150, max_snap_m=150):
    node_ids = np.array(list(node_coords.keys()))
    node_lats = np.array([node_coords[n][0] for n in node_ids])
    node_lons = np.array([node_coords[n][1] for n in node_ids])

    charging_df = pd.read_csv(charging_csv_path, sep=";", encoding="latin-1", header=10)
    charging_df.columns = charging_df.columns.str.strip()

    charging_df = charging_df[[
        "Ladeeinrichtungs-ID",
        "Breitengrad",
        "Längengrad",
        "Nennleistung Ladeeinrichtung [kW]"
    ]].dropna().copy()

    charging_df.columns = ["id", "lat", "lon", "power_kw"]

    charging_df["lat"] = charging_df["lat"].astype(str).str.replace(",", ".").astype(float)
    charging_df["lon"] = charging_df["lon"].astype(str).str.replace(",", ".").astype(float)
    charging_df["power_kw"] = charging_df["power_kw"].astype(str).str.replace(",", ".").astype(float)

    charging_df = charging_df[charging_df["power_kw"] >= min_power_kw].copy()

    print(f"Schnelllader >= {min_power_kw} kW: {len(charging_df):,}")

    charging_nodes = []

    for _, row in charging_df.iterrows():
        dists = haversine_vectorized(row["lat"], row["lon"], node_lats, node_lons)
        idx = np.argmin(dists)

        node_id = int(node_ids[idx])
        snap_dist = float(dists[idx])

        charging_nodes.append({
            "charger_id": row["id"],
            "node": node_id,
            "lat": row["lat"],
            "lon": row["lon"],
            "power_kw": row["power_kw"],
            "snap_dist_m": snap_dist
        })

    charging_nodes_df = pd.DataFrame(charging_nodes)
    charging_nodes_df = charging_nodes_df[charging_nodes_df["snap_dist_m"] <= max_snap_m].copy()

    print(f"Nach Snap-Filter <= {max_snap_m} m: {len(charging_nodes_df):,}")

    charging_hubs_df = charging_nodes_df.groupby("node").agg(
        max_power_kw=("power_kw", "max"),
        n_chargers=("charger_id", "count"),
        min_snap_dist_m=("snap_dist_m", "min")
    ).reset_index()

    print(f"Einzigartige Ladeknoten: {len(charging_hubs_df):,}")

    return charging_nodes_df, charging_hubs_df


if __name__ == "__main__":
    GRAPH_PATH = r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Routenplanung_Gefahrguttransport\models\Berlin\germany_graph_with_accidents_small.pkl"
    POP_GPKG = r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\Umwelftfreundliche_Routenplanung\data\raw\ESTAT_Census_2021_V2.gpkg"
    CHARGING_CSV = r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Operations Research\Ladesaeulenregister_BNetzA_2026-04-22.csv"
    OUTPUT_PATH = "germany_graph_final_small.pkl"

    with open(GRAPH_PATH, "rb") as f:
        data = pickle.load(f)

    edges_df = pd.DataFrame(data["edges"])

    def make_line(row):
        c1 = data["node_coords"].get(row["u"])
        c2 = data["node_coords"].get(row["v"])
        if c1 and c2:
            return LineString([(c1[1], c1[0]), (c2[1], c2[0])])
        return None

    edges_df["geometry"] = edges_df.apply(make_line, axis=1)
    edges_df = edges_df.dropna(subset=["geometry"])
    gdf_roads = gpd.GeoDataFrame(edges_df, geometry="geometry", crs="EPSG:4326")

    # Bevölkerung auf Kanten
    roads_with_pop = map_population_to_edges(gdf_roads, POP_GPKG, buffer_m=50)

    # Ladesäulen auf Knoten
    charging_nodes_df, charging_hubs_df = map_charging_to_nodes(
        data["node_coords"],
        CHARGING_CSV,
        min_power_kw=150,
        max_snap_m=150
    )

    final_edge_cols = [
        "arc_id", "u", "v", "dist", "highway", "name", "tunnel", "length_m",
        "accidents", "weighted_accidents", "accident_score", "accident_score_norm",
        "total_pop", "pop_per_m", "pop_score_norm"
    ]
    final_edge_cols = [c for c in final_edge_cols if c in roads_with_pop.columns]

    export_data = {
        "nodes": data["nodes"],
        "node_coords": data["node_coords"],
        "edges": roads_with_pop[final_edge_cols].to_dict("records"),
        "charging_nodes": charging_nodes_df.to_dict("records"),
        "charging_hubs": charging_hubs_df.to_dict("records"),
    }

    with open(OUTPUT_PATH, "wb") as f:
        pickle.dump(export_data, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"\n✅ Finaler Graph inkl. Ladesäulen: {OUTPUT_PATH}")


Bevölkerungszellen: 1,901,599
✅ Bevölkerung gemappt:
   Kanten mit Bevölkerung > 0: 2,983,134
   Max Pop/m: 83.73


C:\Users\tspol\AppData\Local\Temp\ipykernel_14012\3427989696.py:81: DtypeWarning: Columns (0: Nennleistung Stecker5, 1: Public Key5, 2: Nennleistung Stecker6, 3: Public Key6) have mixed types. Specify dtype option on import or set low_memory=False.
  charging_df = pd.read_csv(charging_csv_path, sep=";", encoding="latin-1", header=10)


Schnelllader >= 150 kW: 21,657
Nach Snap-Filter <= 150 m: 1,638
Einzigartige Ladeknoten: 657

✅ Finaler Graph inkl. Ladesäulen: germany_graph_final_small.pkl


Dijkstra

In [ ]:
import pickle
import pandas as pd
import numpy as np
import networkx as nx
import math

# ============================================
# 1. KONFIGURATION
# ============================================

GRAPH_PATH = r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Routenplanung_Gefahrguttransport\models\Berlin\germany_graph_final_small.pkl"

OUTPUT_CSV = "od_matrix_hazmat_small.csv"
OUTPUT_POINTS_CSV = "mapped_points_small.csv"
OUTPUT_PATHS_PKL = "routes_with_used_nodes_small.pkl"

POINTS = [
    {"id": "DEPOT", "lat": 51.325918, "lon": 12.011645, "type": "depot"},
    {"id": "C1",    "lat": 51.282499, "lon": 9.526584,  "type": "customer"},
    {"id": "C2",    "lat": 51.623978, "lon": 9.936278,  "type": "customer"},
    {"id": "C3",    "lat": 51.488659, "lon": 9.007833,  "type": "customer"},
    {"id": "C4",    "lat": 51.287606, "lon": 8.873039,  "type": "customer"},
    {"id": "C5",    "lat": 51.029800, "lon": 9.421600,  "type": "customer"},
    {"id": "C6",    "lat": 50.861352, "lon": 9.721422,  "type": "customer"},
    {"id": "C7",    "lat": 50.980433, "lon": 10.287236, "type": "customer"},
    {"id": "C8",    "lat": 51.200747, "lon": 10.018705, "type": "customer"},
    {"id": "C9",    "lat": 51.390280, "lon": 10.125665, "type": "customer"},
    {"id": "C10",   "lat": 51.432132, "lon": 9.647465,  "type": "customer"},
]

PROFILES = {
    "shortest": {"lambda_risk": 0.0, "lambda_road": 0.0},
    "safest":   {"lambda_risk": 50.0, "lambda_road": 5.0},
}

# Gewichtung der Risikofaktoren
W_ACC = 0.5
W_POP = 0.5

ONLY_LARGEST_COMPONENT = True

# Durchschnittsgeschwindigkeiten nach Straßentyp
SPEED_KMH = {
    "motorway": 80,
    "motorway_link": 50,
    "trunk": 70,
    "trunk_link": 45,
    "primary": 60,
    "primary_link": 40,
    "secondary": 50,
    "secondary_link": 35,
    "tertiary": 40,
    "tertiary_link": 30,
    "residential": 25,
    "unclassified": 30,
}

# Gefahrgut soll Autobahnen möglichst bevorzugen:
# je höher, desto "schlechter" für Hazmat
ROAD_PENALTY = {
    "motorway": 0.00,
    "motorway_link": 0.05,
    "trunk": 0.08,
    "trunk_link": 0.12,
    "primary": 0.20,
    "primary_link": 0.25,
    "secondary": 0.35,
    "secondary_link": 0.40,
    "tertiary": 0.55,
    "tertiary_link": 0.60,
    "residential": 0.90,
    "unclassified": 0.70,
}

# ============================================
# 2. HILFSFUNKTIONEN
# ============================================

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

def path_signature(path, max_nodes=8):
    if not path:
        return ""
    if len(path) <= max_nodes:
        return "->".join(map(str, path))
    head = "->".join(map(str, path[:4]))
    tail = "->".join(map(str, path[-2:]))
    return f"{head}->...->{tail}"

# ============================================
# 3. GRAPH LADEN
# ============================================

print("Graph laden...")
with open(GRAPH_PATH, "rb") as f:
    data = pickle.load(f)

node_coords = data["node_coords"]
edges = data["edges"]

print(f"  Knoten: {len(node_coords):,}")
print(f"  Kanten: {len(edges):,}")

# ============================================
# 4. NETWORKX GRAPH AUFBAUEN
# ============================================

print("\nGraph aufbauen...")

G = nx.DiGraph()

for edge in edges:
    u = edge["u"]
    v = edge["v"]

    # Distanz immer als Meter -> km
    dist_m = edge.get("dist", edge.get("length_m", 0))
    dist_km = float(dist_m) / 1000.0

    highway = edge.get("highway", "unclassified")
    speed_kmh = SPEED_KMH.get(highway, 30)
    time_h = dist_km / speed_kmh if speed_kmh > 0 else 999999.0

    acc_score = float(edge.get("accident_score_norm", 0))
    pop_score = float(edge.get("pop_score_norm", 0))

    base_risk = W_ACC * acc_score + W_POP * pop_score
    risk_cost = base_risk * dist_km

    road_penalty = ROAD_PENALTY.get(highway, 0.70)
    road_penalty_cost = road_penalty * dist_km

    G.add_edge(
        u, v,
        dist_km=dist_km,
        time_h=time_h,
        speed_kmh=speed_kmh,
        acc_score=acc_score,
        pop_score=pop_score,
        base_risk=base_risk,
        risk_cost=risk_cost,
        road_penalty=road_penalty,
        road_penalty_cost=road_penalty_cost,
        highway=highway,
        name=edge.get("name", ""),
        tunnel=edge.get("tunnel", "no"),
    )

print(f"  Graph: {G.number_of_nodes():,} Knoten, {G.number_of_edges():,} Kanten")

# ============================================
# 5. GRÖSSTE KOMPONENTE
# ============================================

print("\nKonnektivität prüfen...")

if ONLY_LARGEST_COMPONENT and not nx.is_weakly_connected(G):
    components = list(nx.weakly_connected_components(G))
    print(f"  ⚠️ {len(components)} Komponenten gefunden")
    largest = max(components, key=len)
    G = G.subgraph(largest).copy()
    print(f"  → Reduziert auf {G.number_of_nodes():,} Knoten")
else:
    print("  ✓ Graph ist zusammenhängend oder Reduktion deaktiviert")

valid_nodes = set(G.nodes())

# ============================================
# 6. NEAREST NODE
# ============================================

node_list = [n for n in node_coords.keys() if n in valid_nodes]
coords_array = np.array([node_coords[n] for n in node_list])  # (lat, lon)
node_array = np.array(node_list)

def nearest_node(lat, lon):
    dists = np.sqrt(
        (coords_array[:, 0] - lat) ** 2 +
        (coords_array[:, 1] - lon) ** 2
    )
    idx = np.argmin(dists)
    node = int(node_array[idx])
    node_lat, node_lon = coords_array[idx]
    snap_dist_m = haversine_m(lat, lon, node_lat, node_lon)
    return node, snap_dist_m, node_lat, node_lon

# ============================================
# 7. PUNKTE MAPPEN
# ============================================

print("\nPunkte mappen...")

points_df = pd.DataFrame(POINTS)
mapped = points_df.apply(lambda r: nearest_node(r["lat"], r["lon"]), axis=1)

points_df["node"] = [m[0] for m in mapped]
points_df["snap_dist_m"] = [m[1] for m in mapped]
points_df["node_lat"] = [m[2] for m in mapped]
points_df["node_lon"] = [m[3] for m in mapped]

for _, row in points_df.iterrows():
    print(
        f"  {row['id']} ({row['type']}): Node {row['node']} @ "
        f"({row['node_lat']:.4f}, {row['node_lon']:.4f}) | "
        f"Snap-Distanz: {row['snap_dist_m']:.1f} m"
    )

dup_nodes = points_df.groupby("node")["id"].apply(list)
dup_nodes = dup_nodes[dup_nodes.apply(len) > 1]
if len(dup_nodes) > 0:
    print("\n⚠️ Mehrere Punkte liegen auf demselben Graph-Knoten:")
    for node_id, ids in dup_nodes.items():
        print(f"  Node {node_id}: {ids}")

# ============================================
# 8. PFADFUNKTION
# ============================================

print("\nRouten berechnen...")

def make_weight(profile_name):
    lam_risk = PROFILES[profile_name]["lambda_risk"]
    lam_road = PROFILES[profile_name]["lambda_road"]

    def weight_func(u, v, d):
        return d["dist_km"] + lam_risk * d["risk_cost"] + lam_road * d["road_penalty_cost"]
    return weight_func

def compute_path(source, target, profile_name, debug=False):
    lam_risk = PROFILES[profile_name]["lambda_risk"]
    lam_road = PROFILES[profile_name]["lambda_road"]
    weight_func = make_weight(profile_name)

    try:
        distance, path = nx.single_source_dijkstra(
    G,
    source,
    target=target,
    weight=weight_func
)

        total_dist = 0.0
        total_time_h = 0.0
        total_risk_cost = 0.0
        total_road_penalty_cost = 0.0
        highway_km = {}

        for i in range(len(path) - 1):
            e = G[path[i]][path[i + 1]]
            total_dist += e["dist_km"]
            total_time_h += e["time_h"]
            total_risk_cost += e["risk_cost"]
            total_road_penalty_cost += e["road_penalty_cost"]

            hwy = e["highway"]
            highway_km[hwy] = highway_km.get(hwy, 0.0) + e["dist_km"]

        objective_value = (
            total_dist
            + lam_risk * total_risk_cost
            + lam_road * total_road_penalty_cost
        )

        used_nodes = path
        used_coords = [node_coords[n] for n in path if n in node_coords]

        if debug:
            print(f"\nDEBUG {profile_name}:")
            print(f"  Distanz: {total_dist:.3f} km")
            print(f"  Zeit: {total_time_h*60:.2f} min")
            print(f"  Risikokosten: {total_risk_cost:.6f}")
            print(f"  Straßen-Penalty: {total_road_penalty_cost:.6f}")
            print(f"  Objective: {objective_value:.6f}")
            print(f"  Path-Signature: {path_signature(path)}")
            print(f"  Verwendete Knoten: {used_nodes[:20]}")
            if len(used_nodes) > 20:
                print(f"    ... insgesamt {len(used_nodes)} Knoten")
            print("  Straßentyp-Nutzung:")
            for k, v in sorted(highway_km.items(), key=lambda x: -x[1]):
                print(f"    {k}: {v:.3f} km")

        return {
            "reachable": True,
            "dist_km": total_dist,
            "time_h": total_time_h,
            "risk_cost": total_risk_cost,
            "road_penalty_cost": total_road_penalty_cost,
            "objective": objective_value,
            "path_nodes": len(path),
            "used_nodes": used_nodes,
            "used_coords": used_coords,
            "path_signature": path_signature(path),
            "highway_km": highway_km,
        }

    except nx.NetworkXNoPath:
        return {
            "reachable": False,
            "dist_km": np.nan,
            "time_h": np.nan,
            "risk_cost": np.nan,
            "road_penalty_cost": np.nan,
            "objective": np.nan,
            "path_nodes": 0,
            "used_nodes": [],
            "used_coords": [],
            "path_signature": "",
            "highway_km": {},
        }

# ============================================
# 9. OD-MATRIX + ROUTEN
# ============================================

results = []

for _, src in points_df.iterrows():
    for _, dst in points_df.iterrows():
        if src["id"] == dst["id"]:
            continue

        for profile_name in PROFILES.keys():
            debug_flag = (src["id"] == "DEPOT" and dst["id"] == "C1")
            result = compute_path(src["node"], dst["node"], profile_name, debug=debug_flag)

            results.append({
                "from": src["id"],
                "to": dst["id"],
                "from_type": src["type"],
                "to_type": dst["type"],
                "profile": profile_name,
                "dist_km": result["dist_km"],
                "time_min": result["time_h"] * 60 if result["reachable"] else np.nan,
                "risk_cost": result["risk_cost"],
                "road_penalty_cost": result["road_penalty_cost"],
                "objective": result["objective"],
                "path_nodes": result["path_nodes"],
                "reachable": result["reachable"],
                "path_signature": result["path_signature"],
                "used_nodes": result["used_nodes"],
                "used_coords": result["used_coords"],
                "highway_km": result["highway_km"],
            })

            if result["reachable"]:
                print(f"\n  {src['id']} ({src['type']}) → {dst['id']} ({dst['type']}) [{profile_name}]")
                print(f"    Distanz: {result['dist_km']:.3f} km")
                print(f"    Zeit: {result['time_h']*60:.2f} min")
                print(f"    Risikokosten: {result['risk_cost']:.6f}")
                print(f"    Straßen-Penalty: {result['road_penalty_cost']:.6f}")
                print(f"    Objective: {result['objective']:.6f}")
                print(f"    Path-Signature: {result['path_signature']}")
                print(f"    Verwendete Knoten: {result['used_nodes'][:15]}")
                if len(result['used_nodes']) > 15:
                    print(f"    ... insgesamt {len(result['used_nodes'])} Knoten")

# ============================================
# 10. EXPORT
# ============================================

for r in results:
    if r["reachable"]:
        r["used_nodes_str"] = ";".join(map(str, r["used_nodes"]))
        r["used_coords_str"] = ";".join([f"{lat},{lon}" for lat, lon in r["used_coords"]])
        r["highway_km_str"] = ";".join([f"{k}:{v:.3f}" for k, v in r["highway_km"].items()])
    else:
        r["used_nodes_str"] = ""
        r["used_coords_str"] = ""
        r["highway_km_str"] = ""

od_df = pd.DataFrame(results)
od_df.to_csv(OUTPUT_CSV, index=False)
points_df.to_csv(OUTPUT_POINTS_CSV, index=False)

with open(OUTPUT_PATHS_PKL, "wb") as f:
    pickle.dump({
        "mapped_points": points_df.to_dict("records"),
        "routes": results
    }, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"\n✅ OD-Matrix gespeichert: {OUTPUT_CSV}")
print(f"✅ Gemappte Punkte gespeichert: {OUTPUT_POINTS_CSV}")
print(f"✅ Routen mit verwendeten Knoten gespeichert: {OUTPUT_PATHS_PKL}")


Graph laden...
  Knoten: 1,790,173
  Kanten: 3,348,685

Graph aufbauen...
  Graph: 1,790,173 Knoten, 3,348,620 Kanten

Konnektivität prüfen...
  ⚠️ 51340 Komponenten gefunden
  → Reduziert auf 1,346,870 Knoten

Punkte mappen...
  DEPOT (depot): Node 3705572847 @ (51.3259, 12.0117) | Snap-Distanz: 4.5 m
  C1 (customer): Node 4368427171 @ (51.2824, 9.5267) | Snap-Distanz: 14.1 m
  C2 (customer): Node 2568579150 @ (51.6241, 9.9357) | Snap-Distanz: 42.2 m
  C3 (customer): Node 4327298297 @ (51.4888, 9.0080) | Snap-Distanz: 15.5 m
  C4 (customer): Node 338248893 @ (51.2871, 8.8732) | Snap-Distanz: 55.7 m
  C5 (customer): Node 2631436089 @ (51.0300, 9.4216) | Snap-Distanz: 23.9 m
  C6 (customer): Node 267198456 @ (50.8615, 9.7214) | Snap-Distanz: 18.2 m
  C7 (customer): Node 3153778063 @ (50.9804, 10.2873) | Snap-Distanz: 8.4 m
  C8 (customer): Node 13082075043 @ (51.2007, 10.0187) | Snap-Distanz: 2.7 m
  C9 (customer): Node 2623333220 @ (51.3896, 10.1243) | Snap-Distanz: 122.6 m
  C10 (cust

KeyboardInterrupt: 

In [12]:
import pandas as pd
csv_path = r"C:\Users\tspol\OneDrive\Desktop\Digitale Technologien\4.Semester\Routenplanung_Gefahrguttransport\models\Berlin\od_matrix_hazmat.csv"
df = pd.read_csv(csv_path)
# Nur die "sauberen" Spalten behalten
show_cols = ["from", "to", "profile", "dist_km", "time_min", "risk_cost", "objective", "reachable"]
show_cols = [c for c in show_cols if c in df.columns]
df_small = df[show_cols].copy()
print("Tabellarische Übersicht:")
print(df_small.to_string(index=False))
# Distanzmatrix je Profil
for profile in df_small["profile"].unique():
    print(f"\n=== Distanzmatrix: {profile} ===")
    mat_dist = df_small[df_small["profile"] == profile].pivot(
        index="from", columns="to", values="dist_km"
    )
    print(mat_dist.round(2).fillna("-").to_string())
# Zeitmatrix je Profil
for profile in df_small["profile"].unique():
    print(f"\n=== Zeitmatrix: {profile} ===")
    mat_time = df_small[df_small["profile"] == profile].pivot(
        index="from", columns="to", values="time_min"
    )
    print(mat_time.round(1).fillna("-").to_string())
# Risikomatrix je Profil
for profile in df_small["profile"].unique():
    print(f"\n=== Risikomatrix: {profile} ===")
    mat_risk = df_small[df_small["profile"] == profile].pivot(
        index="from", columns="to", values="risk_cost"
    )
    print(mat_risk.round(4).fillna("-").to_string())

Tabellarische Übersicht:
 from    to  profile   dist_km   time_min  risk_cost  objective  reachable
DEPOT    C1 shortest 25.639755  32.833263   0.204247  25.639755       True
DEPOT    C1   safest 34.649188  35.082810   0.128084  72.433359       True
DEPOT    C2 shortest 38.175013  47.651150   0.205252  38.175013       True
DEPOT    C2   safest 45.462246  40.497787   0.156480  75.173943       True
DEPOT    C3 shortest 17.505766  22.278051   0.251141  17.505766       True
DEPOT    C3   safest 18.343749  20.665974   0.230653  54.453559       True
DEPOT    C4 shortest 44.297211  44.176900   0.260912  44.297211       True
DEPOT    C4   safest 45.459078  40.129583   0.124739  70.413662       True
   C1 DEPOT shortest 25.644747  34.603287   0.216192  25.644747       True
   C1 DEPOT   safest 31.643072  32.345477   0.200209  69.374199       True
   C1    C2 shortest 15.393968  28.013251   0.021672  15.393968       True
   C1    C2   safest 23.007100  27.442148   0.024461  56.068025       True
